In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# Define paths
dataset_path = './ExDark/'
preprocessed_train_path = './train/'
preprocessed_test_path = './test/'

# Ensure output directories exist
os.makedirs(preprocessed_train_path, exist_ok=True)
os.makedirs(preprocessed_test_path, exist_ok=True)

# Categories
categories = ["Boat", "Bicycle", "Bus", "Car", "Motorbike"]

# Split ratio for train/test
split_ratio = 0.8

# Custom preprocessing function
def custom_preprocessing(image):
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)

    image = cv2.resize(image, (224, 224))
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    contrast_img = clahe.apply(gray)
    contrast_img = cv2.cvtColor(contrast_img, cv2.COLOR_GRAY2BGR)

    denoised_img = cv2.fastNlMeansDenoisingColored(contrast_img, None, 10, 10, 7, 21)

    kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
    sharpened_img = cv2.filter2D(denoised_img, -1, kernel)

    hsv = cv2.cvtColor(sharpened_img, cv2.COLOR_BGR2HSV)
    hsv[:, :, 2] = cv2.equalizeHist(hsv[:, :, 2])
    hist_eq_img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    final_img = hist_eq_img / 255.0
    final_img = final_img.astype(np.float32)
    
    return final_img

# Function to preprocess and save images
def preprocess_and_save_images(category):
    # Create directories for this category
    os.makedirs(os.path.join(preprocessed_train_path, category), exist_ok=True)
    os.makedirs(os.path.join(preprocessed_test_path, category), exist_ok=True)

    category_path = os.path.join(dataset_path, category)
    images = os.listdir(category_path)
    np.random.shuffle(images)

    split_point = int(len(images) * split_ratio)
    train_images = images[:split_point]
    test_images = images[split_point:]

    for image_name in tqdm(train_images, desc=f"Preprocessing train {category}"):
        image = cv2.imread(os.path.join(category_path, image_name))
        preprocessed_image = custom_preprocessing(image)
        save_path = os.path.join(preprocessed_train_path, category, image_name)
        cv2.imwrite(save_path, (preprocessed_image * 255).astype(np.uint8))  # Save as uint8

    for image_name in tqdm(test_images, desc=f"Preprocessing test {category}"):
        image = cv2.imread(os.path.join(category_path, image_name))
        preprocessed_image = custom_preprocessing(image)
        save_path = os.path.join(preprocessed_test_path, category, image_name)
        cv2.imwrite(save_path, (preprocessed_image * 255).astype(np.uint8))  # Save as uint8

# Preprocess all images
for category in categories:
    preprocess_and_save_images(category)

print("All images have been preprocessed and saved.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Function to show images
def show_image(title, img):
    plt.figure(figsize=(5, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

# Custom preprocessing function
def custom_preprocessing(image):
    # Convert the image to uint8 if it's not already
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)
        
    show_image("Original Image", image)

    # Step 1: Resize the image
    image = cv2.resize(image, (224, 224))
    show_image("Resized Image", image)

    # Step 2: Contrast Adjustment (CLAHE)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    contrast_img = clahe.apply(gray)
    contrast_img = cv2.cvtColor(contrast_img, cv2.COLOR_GRAY2BGR)
    show_image("Contrast Adjusted Image (CLAHE)", contrast_img)

    # Step 3: Denoising
    denoised_img = cv2.fastNlMeansDenoisingColored(contrast_img, None, 10, 10, 7, 21)
    show_image("Denoised Image", denoised_img)

    # Step 4: Image Sharpening
    kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
    sharpened_img = cv2.filter2D(denoised_img, -1, kernel)
    show_image("Sharpened Image", sharpened_img)

    # Step 5: Histogram Equalization (on the V channel of HSV)
    hsv = cv2.cvtColor(sharpened_img, cv2.COLOR_BGR2HSV)
    hsv[:, :, 2] = cv2.equalizeHist(hsv[:, :, 2])
    hist_eq_img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    show_image("Histogram Equalized Image", hist_eq_img)

    # Final Step: Normalization (Rescale pixels to [0, 1] and convert to float32)
    final_img = hist_eq_img / 255.0
    final_img = final_img.astype(np.float32)
    show_image("Final Preprocessed Image", (final_img * 255).astype(np.uint8))  # Convert back to uint8 for display

    return final_img

# Simulate preprocessing on a sample image
sample_image_path = os.path.join(dataset_path, categories[0], os.listdir(os.path.join(dataset_path, categories[0]))[0])  # Replace with your actual image path
sample_image = cv2.imread(sample_image_path)
preprocessed_image = custom_preprocessing(sample_image)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Set mixed precision policy for faster training
from tensorflow.keras.mixed_precision import set_global_policy, Policy
set_global_policy(Policy('mixed_float16'))

# Data generators for preprocessed data
train_datagen = ImageDataGenerator(rescale=1./255, 
                                   brightness_range=[0.2, 1.0], 
                                   shear_range=0.2, 
                                   zoom_range=0.2, 
                                   horizontal_flip=True, 
                                   fill_mode='nearest')

test_datagen = ImageDataGenerator(rescale=1./255)

# Load preprocessed training data
train_generator = train_datagen.flow_from_directory(
    preprocessed_train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    classes=categories
)

# Load preprocessed testing data
test_generator = test_datagen.flow_from_directory(
    preprocessed_test_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    classes=categories
)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import models, layers

# Create the model using MobileNetV2 for faster and more efficient training
print("Building the model...")
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze the base model initially

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),  # Added Dropout for regularization
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5),  # Added Dropout for regularization
    layers.Dense(len(categories), activation='softmax', dtype='float32')  # Keep this layer in float32
])

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Print the model summary
model.summary()
print("Model built and compiled.")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Callbacks for early stopping and saving the best model
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True)  # Updated to .keras extension
]

# Train the top layers first
print("Training the top layers...")
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 32,
    validation_data=test_generator,
    validation_steps=test_generator.samples // 32,
    epochs=10,
    callbacks=callbacks)
print("Top layers training completed.")

In [ ]:
base_model.trainable = True # Enable parent before selectively unfreezing.
# Fine-tune the base model by unfreezing only the last few layers
print("Fine-tuning the model...")
for layer in base_model.layers[:-30]:  # Unfreeze only the last 30 layers
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

# Re-compile the model with a lower learning rate for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Early stopping criteria more strict to prevent long training times
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

# Reduce the number of epochs for fine-tuning
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 32,
    validation_data=test_generator,
    validation_steps=test_generator.samples // 32,
    epochs=10,
    callbacks=callbacks)
print("Fine-tuning completed.")


In [ ]:
import matplotlib.pyplot as plt

# Plot the training and validation accuracy and loss at each epoch
print("Plotting the training results...")
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Over Epochs')
plt.legend()

plt.show()
print("Training results plotted.")